In [1]:
!pip install roboflow ultralytics pyyaml -q
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("✅ Ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.1 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
CUDA: True
GPU : Tesla T4
✅ Ready!


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY_HERE")  # Replace with your actual API key

print("Downloading Dataset 1 — Indian + General food...")
ds1 = rf.workspace("object-detection-vpvcm").project("food-images-2lpjg").version(1).download("yolov8")

print("Downloading Dataset 2 — Indian food expanded...")
ds2 = rf.workspace("project-z0tql").project("indian-food-detection-kzw9g").version(1).download("yolov8")

print("Downloading Dataset 3 — Food tracker (healthy foods)...")
ds3 = rf.workspace("foodtrackerdataset").project("food-tracker-yolo").version(13).download("yolov8")

print("\n✅ All 3 datasets downloaded!")
print(f"DS1: {ds1.location}")
print(f"DS2: {ds2.location}")
print(f"DS3: {ds3.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Food-Images-1 in yolov8:: 100%|██████████| 9700/9700 [00:01<00:00, 7862.57it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Indian-food-detection-1 in yolov8:: 100%|██████████| 18118/18118 [00:02<00:00, 8214.36it/s] 


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to food-tracker-yolo-13 in yolov8:: 100%|██████████| 59496/59496 [00:07<00:00, 8453.23it/s] 



✅ All 3 datasets downloaded!
DS1: /kaggle/working/Food-Images-1
DS2: /kaggle/working/Indian-food-detection-1
DS3: /kaggle/working/food-tracker-yolo-13


In [3]:
import os, shutil, yaml, random
from pathlib import Path

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

y1 = load_yaml(f"{ds1.location}/data.yaml")
y2 = load_yaml(f"{ds2.location}/data.yaml")
y3 = load_yaml(f"{ds3.location}/data.yaml")

c1 = [c.lower().strip() for c in y1.get('names', [])]
c2 = [c.lower().strip() for c in y2.get('names', [])]
c3 = [c.lower().strip() for c in y3.get('names', [])]

# Merge unique classes
seen, all_classes = set(), []
for cls in c1 + c2 + c3:
    if cls not in seen:
        seen.add(cls)
        all_classes.append(cls)

print(f"DS1: {len(c1)} classes | DS2: {len(c2)} classes | DS3: {len(c3)} classes")
print(f"Total unique classes: {len(all_classes)}")

# Create merged folders
merged = Path("/kaggle/working/merged_v2")
for split in ['train', 'valid']:
    (merged / split / 'images').mkdir(parents=True, exist_ok=True)
    (merged / split / 'labels').mkdir(parents=True, exist_ok=True)

def remap_and_copy(ds_location, src_classes, split, prefix, all_classes):
    for split_name in [split, 'valid' if split == 'val' else split, 'train']:
        img_dir = Path(ds_location) / split_name / 'images'
        lbl_dir = Path(ds_location) / split_name / 'labels'
        if img_dir.exists():
            break
    else:
        print(f"  ⚠️  No {split} in {ds_location}")
        return 0, 0

    n_img = n_lbl = 0
    for img_file in img_dir.glob('*'):
        if img_file.suffix.lower() not in ['.jpg','.jpeg','.png','.webp']:
            continue
        shutil.copy(img_file, merged/split/'images'/(prefix+img_file.name))
        n_img += 1
        lbl_file = lbl_dir / (img_file.stem + '.txt')
        if lbl_file.exists():
            new_lines = []
            with open(lbl_file) as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    parts = line.split()
                    old_id = int(parts[0])
                    if old_id >= len(src_classes): continue
                    cls_name = src_classes[old_id]
                    if cls_name not in all_classes: continue
                    new_id = all_classes.index(cls_name)
                    new_lines.append(f"{new_id} {' '.join(parts[1:])}")
            if new_lines:
                dst = merged/split/'labels'/(prefix+img_file.stem+'.txt')
                with open(dst, 'w') as f:
                    f.write('\n'.join(new_lines))
                n_lbl += 1
    return n_img, n_lbl

datasets = [(ds1.location,c1,'ds1_'),(ds2.location,c2,'ds2_'),(ds3.location,c3,'ds3_')]
for split in ['train','valid']:
    print(f"\n── {split.upper()} ──")
    for loc, classes, prefix in datasets:
        ni, nl = remap_and_copy(loc, classes, split, prefix, all_classes)
        print(f"  {prefix}: {ni} images, {nl} labels")

# Write data.yaml
merged_yaml = {
    'path':  str(merged),
    'train': 'train/images',
    'val':   'valid/images',
    'nc':    len(all_classes),
    'names': all_classes,
}
with open(merged/'data.yaml','w') as f:
    yaml.dump(merged_yaml, f, default_flow_style=False)

total_train = len(list((merged/'train'/'images').glob('*')))
total_val   = len(list((merged/'valid'/'images').glob('*')))
print(f"\n✅ Merged dataset ready!")
print(f"Train: {total_train} | Val: {total_val} | Classes: {len(all_classes)}")

DS1: 59 classes | DS2: 17 classes | DS3: 35 classes
Total unique classes: 107

── TRAIN ──
  ds1_: 4113 images, 4007 labels
  ds2_: 7547 images, 6506 labels
  ds3_: 28932 images, 28848 labels

── VALID ──
  ds1_: 424 images, 406 labels
  ds2_: 1053 images, 929 labels
  ds3_: 530 images, 530 labels

✅ Merged dataset ready!
Train: 40592 | Val: 2007 | Classes: 107


In [4]:
def trim_dataset(split_dir, max_images):
    img_dir = Path(split_dir) / 'images'
    lbl_dir = Path(split_dir) / 'labels'
    all_imgs = list(img_dir.glob('*'))
    if len(all_imgs) <= max_images:
        print(f"  {Path(split_dir).name}: {len(all_imgs)} — no trim needed")
        return
    keep   = set(random.sample(all_imgs, max_images))
    remove = [f for f in all_imgs if f not in keep]
    for img in remove:
        img.unlink()
        lbl = lbl_dir / (img.stem + '.txt')
        if lbl.exists(): lbl.unlink()
    print(f"  {Path(split_dir).name}: trimmed to {max_images} images")

print("Trimming to safe size...")
trim_dataset(merged/'train', max_images=10000)
trim_dataset(merged/'valid', max_images=2000)

final_train = len(list((merged/'train'/'images').glob('*')))
final_val   = len(list((merged/'valid'/'images').glob('*')))
print(f"\n✅ Final — Train: {final_train} | Val: {final_val}")

Trimming to safe size...
  train: trimmed to 10000 images
  valid: trimmed to 2000 images

✅ Final — Train: 10000 | Val: 2000


In [5]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Training on: {'GPU ✅' if device == 0 else 'CPU ⚠️'}")

model = YOLO('yolov8s.pt')

results = model.train(
    data          = str(merged/'data.yaml'),
    epochs        = 100,
    imgsz         = 640,
    batch         = 16,
    name          = 'fitai_v2',
    project       = '/kaggle/working/runs',
    patience      = 20,
    device        = device,
    augment       = True,
    mosaic        = 1.0,
    mixup         = 0.15,
    copy_paste    = 0.1,
    fliplr        = 0.5,
    flipud        = 0.1,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.5,
    lr0           = 0.01,
    lrf           = 0.01,
    warmup_epochs = 5,
    weight_decay  = 0.0005,
    save          = True,
    save_period   = 10,
    workers       = 4,
)
print("✅ Training complete!")

Training on: GPU ✅
Ultralytics 8.4.39 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/merged_v2/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fitai_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overla

In [6]:
metrics = model.val()
print(f"\n{'='*45}")
print(f"🏆 FITAI v2 FINAL RESULTS")
print(f"{'='*45}")
print(f"mAP50    : {metrics.box.map50:.3f}  (target >0.65)")
print(f"mAP50-95 : {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall   : {metrics.box.mr:.3f}")
print(f"Classes  : {len(all_classes)}")
print(f"{'='*45}")
if metrics.box.map50 >= 0.65:
    print("🔥 Model is production ready!")
elif metrics.box.map50 >= 0.55:
    print("✅ Good — increase epochs for better results")
else:
    print("⚠️ Try more epochs or more data")

Ultralytics 8.4.39 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,166,993 parameters, 0 gradients, 28.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1236.0±327.2 MB/s, size: 52.9 KB)
val: Scanning /kaggle/working/merged_v2/valid/labels.cache... 1858 images, 142 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2000/2000 762.6Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 107, len(boxes) = 2970. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 125/125 4.7it/s 26.9s0.2s
                   all       2000       2970      0.649      0.619      0.631      0.498
             aloo gobi         29         29      0.391      0.655      0.517      0.274
            aloo ma

In [7]:
import shutil
from pathlib import Path

# Find best.pt
best_pt = Path('/kaggle/working/runs/fitai_v2/weights/best.pt')

# Copy to output folder
shutil.copy(best_pt, '/kaggle/working/best_fitai_v2.pt')
print(f"✅ best.pt saved to Kaggle output!")
print(f"📁 Go to Output panel → Download best_fitai_v2.pt")
print(f"📁 Then place it in → fitnessai/Model1_FoodDetection/best.pt")

# Also print all classes for reference
print(f"\n📋 All {len(all_classes)} classes your model can detect:")
for i, cls in enumerate(all_classes):
    print(f"  {i:3d}. {cls}")

✅ best.pt saved to Kaggle output!
📁 Go to Output panel → Download best_fitai_v2.pt
📁 Then place it in → fitnessai/Model1_FoodDetection/best.pt

📋 All 107 classes your model can detect:
    0. aloo gobi
    1. aloo matar
    2. aloo methi
    3. apple
    4. bhindi masala
    5. biryani
    6. boiled egg
    7. bread
    8. burger
    9. butter chicken
   10. chai
   11. chicken curry
   12. chicken tikka
   13. chicken wings
   14. cholay
   15. chutney
   16. cucumber
   17. daal
   18. dal makhni
   19. donuts
   20. dosa
   21. dumplings
   22. french fries
   23. french toast
   24. fried egg
   25. gulab jamun
   26. ice cream
   27. kadhi pakora
   28. kheer
   29. khichdi
   30. kulfi
   31. lasagna
   32. mac and cheese
   33. naan
   34. nachos
   35. omelette
   36. onion rings
   37. pakora
   38. palak paneer
   39. pancakes
   40. paratha
   41. pizza
   42. potato cutlets
   43. raita
   44. rasgulla
   45. red beans
   46. rice
   47. roti
   48. salad
   49. samosa
   5